In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import csv
import pandas as pd


In [3]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/NLP_Ctrl-Alt-Elite/data/dataset.csv')

In [4]:
df.head()

,label,text
0,1,Congratulations! You've been selected for a lu...
1,1,URGENT: Your account has been compromised. Cli...
2,1,You've won a free iPhone! Claim your prize by ...
3,1,Act now and receive a 50% discount on all purc...
4,1,Important notice: Your subscription will expir...


In [5]:
df2 = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/NLP_Ctrl-Alt-Elite/data/validation_dataset.csv')

In [6]:
df2.head()

,Email Text,Email Type
0,"Dear Jordan, your subscription has been succes...",Safe Email
1,"Dear Casey, thank you for your purchase. Your ...",Safe Email
2,Congratulations! You've won a $3000 gift card....,Phishing Email
3,You have a new secure message from your bank. ...,Phishing Email
4,Your package delivery is pending. Please provi...,Phishing Email


01 - Data Collection & Description

## Setup
Run the cell below first. It detects whether you're in **Google Colab** or
running **locally in VS Code**, and gets the environment ready either way
(clones the repo in Colab, installs requirements, downloads NLTK data, and
adds `src/` to the path so `pipeline.py` can be imported).

In [7]:
# ============================================================
# SETUP CELL - run this first, every time
# Works both locally (VS Code / Jupyter) and in Google Colab
# ============================================================
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/Sandaru17513/NLP_Ctrl-Alt-Elite.git"
    REPO_DIR = "NLP_Ctrl-Alt-Elite"

    if not os.path.exists(REPO_DIR):
        get_ipython().system(f"git clone {REPO_URL}")
    os.chdir(f"{REPO_DIR}/notebooks")

    get_ipython().system("pip install -q -r ../requirements.txt")

    # Data files are large - if they were not committed to the repo,
    # upload them here once per Colab session.
    if not os.path.exists("../data/data.csv"):
        print("data/data.csv not found in the cloned repo.")
        print("Option A: git add + commit + push the CSVs from your")
        print("          local machine so they come down with the clone.")
        print("Option B: uncomment the lines below to upload manually.")
        # from google.colab import files
        # uploaded = files.upload()   # select data.csv + validation_dataset.csv
        # os.makedirs("../data", exist_ok=True)
        # for fname in uploaded:
        #     os.rename(fname, f"../data/{fname}")
else:
    print("Running locally (VS Code / Jupyter). Using existing .venv environment.")

import nltk
for pkg in ("punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"):
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass

sys.path.append(os.path.abspath("../src"))
print("IN_COLAB =", IN_COLAB)
print("Working directory:", os.getcwd())


Cloning into 'NLP_Ctrl-Alt-Elite'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 85 (delta 26), reused 74 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (85/85), 33.32 MiB | 14.32 MiB/s, done.
Resolving deltas: 100% (26/26), done.
data/data.csv not found in the cloned repo.
Option A: git add + commit + push the CSVs from your
          local machine so they come down with the clone.
Option B: uncomment the lines below to upload manually.
IN_COLAB = True
Working directory: /content/NLP_Ctrl-Alt-Elite/notebooks


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 1.1 Load and inspect both datasets

In [9]:
import pandas as pd
import numpy as np

df_train = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/NLP_Ctrl-Alt-Elite/data/dataset.csv')
df_val = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/NLP_Ctrl-Alt-Elite/data/validation_dataset.csv')

print('=== TRAINING DATASET (data.csv) ===')
print(f'Shape: {df_train.shape}')
print(f'Columns: {df_train.columns.tolist()}')
print(df_train.head())
print(df_train.dtypes)
print('Nulls per column:')
print(df_train.isnull().sum())
print(df_train['label'].value_counts())
print(df_train['label'].value_counts(normalize=True) * 100)

=== TRAINING DATASET (data.csv) ===
Shape: (45155, 2)
Columns: ['label', 'text']
   label                                               text
0      1  Congratulations! You've been selected for a lu...
1      1  URGENT: Your account has been compromised. Cli...
2      1  You've won a free iPhone! Claim your prize by ...
3      1  Act now and receive a 50% discount on all purc...
4      1  Important notice: Your subscription will expir...
label     int64
text     object
dtype: object
Nulls per column:
label    0
text     0
dtype: int64
label
0    22584
1    22571
Name: count, dtype: int64
label
0    50.014395
1    49.985605
Name: proportion, dtype: float64


In [10]:
print('=== VALIDATION DATASET (validation_dataset.csv) ===')
print(f'Shape: {df_val.shape}')
print(f'Columns: {df_val.columns.tolist()}')
print(df_val.head())
print(df_val.isnull().sum())
print(df_val['Email Type'].value_counts())

=== VALIDATION DATASET (validation_dataset.csv) ===
Shape: (2000, 2)
Columns: ['Email Text', 'Email Type']
                                          Email Text      Email Type
0  Dear Jordan, your subscription has been succes...      Safe Email
1  Dear Casey, thank you for your purchase. Your ...      Safe Email
2  Congratulations! You've won a $3000 gift card....  Phishing Email
3  You have a new secure message from your bank. ...  Phishing Email
4  Your package delivery is pending. Please provi...  Phishing Email
Email Text    0
Email Type    0
dtype: int64
Email Type
Safe Email        1000
Phishing Email    1000
Name: count, dtype: int64


### 1.2 Standardize column names (critical step)
The two datasets use different column names. Rename them so the whole
pipeline downstream works on both without special-casing.

In [11]:
df_train = df_train.rename(columns={'text': 'email_text', 'label': 'email_label'})

df_val = df_val.rename(columns={'Email Text': 'email_text', 'Email Type': 'email_label'})
df_val['email_label'] = df_val['email_label'].map({
    'Safe Email': 'ham',
    'Phishing Email': 'spam'
})

print('Training labels:', df_train['email_label'].unique())
print('Validation labels:', df_val['email_label'].unique())

df_train.to_csv('../data/train_renamed.csv', index=False)
df_val.to_csv('../data/val_renamed.csv', index=False)
print('Saved: train_renamed.csv, val_renamed.csv')

Training labels: [1 0]
Validation labels: ['ham' 'spam']
Saved: train_renamed.csv, val_renamed.csv
